# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [75]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
import chromadb
import json
from pydantic import BaseModel, Field
from tavily import TavilyClient
from datetime import datetime
from typing import List
from lib.messages import BaseMessage

In [76]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [77]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game
@tool
def retrieve_game(query: str):
    """
    Semantic searchs the most relevant game entries in the local vector database.
    args:
        query (str): A question about a game or the game industry.

    Returns
        A list of the most relevant matches, each containing:
        - Platform
        - Name
        - YearOfRelease
        - Description
    """

    db_path = os.path.abspath("chromadb")
    chroma_client = chromadb.PersistentClient(path=db_path)
    collection = chroma_client.get_collection("udaplay")

    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
        })

    return retrieved



In [78]:
results = retrieve_game("Which Mario game was the first 3D platformer?")
print(results)

[{'Platform': 'PlayStation 2', 'Name': 'Grand Theft Auto: San Andreas', 'YearOfRelease': 2004, 'Description': "An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson."}, {'Platform': 'PlayStation 5', 'Name': "Marvel's Spider-Man 2", 'YearOfRelease': 2023, 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.'}, {'Platform': 'PlayStation 3', 'Name': 'Gran Turismo 5', 'YearOfRelease': 2010, 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.'}]


#### Evaluate Retrieval Tool

In [79]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationResult(BaseModel):
    useful: bool = Field(
        ...,
        description="Whether the retrieved documents are sufficient to answer the question."
    )
    description: str = Field(
        ...,
        description="What the model thinks about the quality of the retrieved documents."
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: list):
    """
    Evaluates the usability of the retrieved documents to answer the user's question.
    
    Args:
        question (str): The original question from the user.
        retrieved_docs (list): A list of retrieved documents most similar to the user query in the Vector Database.

    Returns:
        dict: A dictionary containing:
            - useful (bool): Whether the documents are useful to answer the question.
            - description (str): A detailed explanation about the evaluation result.
    """

    if not question or not isinstance(question, str):
        return {
            "useful": False,
            "description": "No valid question was provided."
        }

    if not retrieved_docs:
        return {
            "useful": False,
            "description": (
                f"The query '{question}' returned no documents. "
                "There is not enough evidence to answer the question reliably."
            )
        }

    formatted_docs = []
    for i, doc in enumerate(retrieved_docs, start=1):
        if isinstance(doc, dict):
            formatted_docs.append(
                f"{i}. {doc.get('Name', 'Unknown')} | "
                f"Platform: {doc.get('Platform', 'N/A')} | "
                f"Year: {doc.get('YearOfRelease', 'N/A')} | "
                f"Description: {doc.get('Description', doc.get('content', ''))}"
            )
        else:
            formatted_docs.append(str(doc))


    prompt = f"""
    You are evaluating a retrieval system for a video game research assistant.

    User Question:
    "{question}"

    Retrieved Documents:
    {json.dumps(formatted_docs, ensure_ascii=False)}

    Return ONLY valid JSON in this exact format:
    {{
      "useful": true,
      "description": "short explanation"
    }}

    Instructions:
    - "useful" must be either true or false
    - "description" must explain clearly if the documents are enough to answer the question
    - If the documents are weak or missing facts, say that clearly
    - Do not include anything outside the JSON
    """
                                                                                        
    llm = LLM(api_key=OPENAI_API_KEY)
    response = llm.invoke(prompt, response_format=EvaluationResult)

    if hasattr(response, "parsed") and response.parsed is not None:
        evaluation_result = response.parsed
    else:
        # fallback if the SDK returns plain text
        raw = (response.content or "").strip()
        if raw.startswith("```"):
            raw = raw.strip("`")
            if raw.lower().startswith("json"):
                raw = raw[4:].strip()

        try:
            evaluation_result = EvaluationResult.model_validate_json(raw)
        except Exception:
            try:
                parsed = json.loads(raw)
                evaluation_result = EvaluationResult(**parsed)
            except Exception:
                return {
                    "useful": False,
                    "description": "The model did not return a valid evaluation result."
                }

    return {
        "useful": evaluation_result.useful,
        "description": evaluation_result.description
    }

In [80]:
retrieved = retrieve_game("When was the Spider-Man game released?")

evaluation = evaluate_retrieval(
    question="When was the Spider-Man game released?",
    retrieved_docs=retrieved
)

print(evaluation)

{'useful': True, 'description': "The retrieved documents include information about 'Marvel's Spider-Man 2', which was released in 2023. Although it does not provide the release date for the original Spider-Man game, it is still relevant to the user's inquiry about Spider-Man games."}


#### Game Web Search Tool

In [81]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str):
    """
    Search the web for current information about games when the local vector DB is not enough.

    Args:
        question (str): A question about a game or the video game industry.

    Returns:
        list: A normalized list of search result dictionaries with:
            - title
            - url
            - content
    """
    if not question or not isinstance(question, str):
        return []

    try:
        client = TavilyClient(api_key=TAVILY_API_KEY)
        search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
        )

        formatted_results = {
            "answer": search_result.get("answer", ""),
            "results": search_result.get("results", []),
            "search_metadata": {
                "timestamp": datetime.now().isoformat(),
                "query": question
            }
        }

        return formatted_results

    except Exception as e:
        return [
            {
                "answer": "Search failed",
                "results": f"Web search failed: {str(e)}",
                "search_metadata": {
                    "timestamp": datetime.now().isoformat(),
                    "query": question
                }
            }
        ]

In [82]:
results = game_web_search("When was GTA6 released?")
print(results)

{'answer': 'Grand Theft Auto 6 is set to release on November 19, 2026, for PlayStation 5 and Xbox Series X|S. The game will feature a single-player experience in the fictional state of Leonida, inspired by Florida.', 'results': [{'url': 'https://www.bbc.com/news/articles/c9q20djz4wno', 'title': "GTA 6 - all you need to know about Rockstar's blockbuster ...", 'content': '## When is GTA 6 coming out?\n\nGTA 6 will be released on 19 November - and with Rockstar recently opening up pre-orders, this date seems all but certain.\n\nBut it was delayed twice previously, from autumn 2025 and May 2026.\n\nIn a statement when the second delay was announced, Rockstar said it needed extra months to finish the game with the level of polish fans had come to "expect and deserve".\n\nRockstar said players who pre-order will be able to begin pre-loading it on their consoles on 12 November to ensure they are able to play at launch.\n\n## How much will GTA 6 cost and what can I play it on?\n\nThe sixth gam

### Agent

In [83]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
    
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are UdaPlay, an AI research agent for the video game industry. "
        "Use the local database first. If the retrieved documents are not sufficient, "
        "use web search. Provide concise, factual answers and mention uncertainty when needed."
    ),
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.2
)

In [84]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

run = agent.invoke("Which game was the first 3D platformer in the Mario series?")
final_state = run.get_final_state()

print_messages(final_state["messages"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
 -> (role = system, content = You are UdaPlay, an AI research agent for the video game industry. Use the local database first. If the retrieved documents are not sufficient, use web search. Provide concise, factual answers and mention uncertainty when needed., tool_calls = None)
 -> (role = user, content = Which game was the first 3D platformer in the Mario series?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageToolCall(id='call_J9NuFMi41p4fq7TxOqFWfQfx', function=Function(arguments='{"query":"first 3D platformer in the Mario series"}', name='retrieve_game'), type='function')])
 -> (role = tool, co

In [86]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

for question in questions:
    print(f"\nQuestion: {question}")
    run = agent.invoke(question)
    final_state = run.get_final_state()
    print_messages(final_state["messages"])
    print("-" * 50)


Question: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
 -> (role = system, content = You are UdaPlay, an AI research agent for the video game industry. Use the local database first. If the retrieved documents are not sufficient, use web search. Provide concise, factual answers and mention uncertainty when needed., tool_calls = None)
 -> (role = user, content = Which game was the first 3D platformer in the Mario series?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageToolCall(id='call_J9NuFMi41p4fq7TxOqFWfQfx', function=Function(arguments='{"query":"first 3D platformer in the Mario series"}', name='retrieve_game'), type='function')])
 -> (role = tool, content = "[{'Platform': 'PlayStation 3', 'Name': 'Gran Turismo 5', 'YearOfRelease': 2010, 'Description': 'A comprehensive racin

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes